## 1. Customer Lifetime Value (CLV):
### Goal: 
Estimate how much total revenue a customer will generate over their entire relationship with the
business.
### Why it matters: 
Helps prioritize high-value customers, plan marketing budgets wisely, and improve
retention strategies.
### How to do it:
- Use order_items and order_item_options to compute revenue per order.
- Aggregate total spend per customer_id.
### Group CLV values (for tagging):
- High CLV: Top 20% customers
- Medium CLV: Mid 60%
- Low CLV: Bottom 20%

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_fact_order = spark.read.table('global_partner_project.gold.fact_order')

### Revenue per Order

In [0]:
agg_df_order = df_fact_order.groupBy(['order_id']).agg(F.round(F.sum('item_price'),2).alias('order_amount')).orderBy('order_amount',ascending=False)
#agg_df_order.display()

### Aggregate total spend per customer_id

In [0]:
agg_df_customer = df_fact_order.groupBy(['user_id']).agg(F.round(F.sum('item_price'),2).alias('order_amount')).orderBy('order_amount',ascending=False)
#agg_df_customer.display()

## CLV
### percent_rank to find top 20, 80, low 20

In [0]:
clv_window = Window.orderBy(F.col('order_amount').desc(),F.col('user_id'))

agg_df_customer = agg_df_customer.filter(F.col('user_id')!='NA')
agg_df_customer_clv = (agg_df_customer.withColumn('CLV',F.percent_rank().over(clv_window))
          .withColumn('clv_segment',F.when(F.col('CLV') <= 0.2,'High CLV')
                                .when((F.col('CLV') > 0.2) & (F.col('CLV') <= 0.8),'Medium CLV')
                                .otherwise('Low CLV'))
          )
agg_df_customer_clv = agg_df_customer_clv.drop('CLV')

#agg_df_customer_clv.display()

In [0]:
%sql
DROP TABLE IF EXISTS global_partner_project.mart.customer_lifetime_value

In [0]:
agg_df_customer_clv.write.mode('append').saveAsTable('global_partner_project.mart.customer_lifetime_value')